# GNN development workbench

This notebook is a thin, executable control panel over the repository's main builders, trainer, and evaluator. It does not duplicate their logic. Choose `familyowl` or `2wiki`, optionally rebuild the fixed train/dev/test sample, inspect coverage and candidate rankings, then train and evaluate.

In [7]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "training").exists():
    ROOT = ROOT.parent
assert (ROOT / "training").exists(), (
    f"Run this notebook from the repository or notebooks directory: {ROOT}"
)

# ---- Controls ----
DATASET = "familyowl"  # 'familyowl' or '2wiki'
REBUILD_SPLITS = False
TRAIN_MODEL = True
EVALUATE_MODEL = True
RUN_NAME = "graded_listwise"
EPOCHS = 5

print("Repository:", ROOT)
print("Dataset:", DATASET)

Repository: c:\Users\julie\github\PhD\SAGE-QA
Dataset: familyowl


## Commands that call the maintained pipeline

Set a control above to `True` and run this cell. The 2Wiki LLM extraction needs the API settings used by `utils/llm_client.py`. Extraction cache entries are versioned, so changing the extractor rebuilds stale entries while retaining compatible ones.

In [8]:
PYTHON = sys.executable
FAMILY_DIR = ROOT / "data/development/gnn_rebuilt/FamilyOWL_1hop"
WIKI_DIR = ROOT / "data/development/gnn_rebuilt/2WikiMultiHopQA"
DATA_DIR = FAMILY_DIR if DATASET == "familyowl" else WIKI_DIR
CHECKPOINT_DIR = ROOT / "checkpoints/development/workbench" / f"{DATASET}_{RUN_NAME}"
DETAILS_DIR = ROOT / "outputs/development_runs/workbench" / f"{DATASET}_{RUN_NAME}"


def run(command):
    print(">", subprocess.list2cmdline([str(x) for x in command]))
    subprocess.run([str(x) for x in command], cwd=ROOT, check=True)


family_build = [
    PYTHON,
    "data/build_subgraph_training_data.py",
    "--input-json",
    "data/raw/family/FamilyOWL_1hop.json",
    "--output-dir",
    "data/development/gnn_rebuilt",
    "--selection-manifest",
    "data/development/gnn_samples/familyowl_1hop/sample_manifest.json",
    "--max-subgraph-size",
    "7",
    "--max-context-units",
    "40",
    "--max-negative-per-example",
    "200",
    "--candidate-beam-width",
    "96",
    "--max-candidate-subgraphs",
    "320",
]
wiki_build = [
    PYTHON,
    "data_processing/build_2wiki_subgraph_dataset.py",
    "--train-file",
    "data/raw/2WikiMultihopQA/data/train-00000-of-00002.parquet",
    "data/raw/2WikiMultihopQA/data/train-00001-of-00002.parquet",
    "--dev-file",
    "data/raw/2WikiMultihopQA/data/validation-00000-of-00001.parquet",
    "--output-dir",
    "data/development/gnn_rebuilt/2WikiMultiHopQA",
    "--selection-manifest",
    "data/development/gnn_samples/2wiki/sample_manifest.json",
    "--max-kg-candidate-triples",
    "30",
    "--max-subgraph-size",
    "4",
    "--max-kg-bridge-triples",
    "64",
    "--kg-construction-backend",
    "llm",
    "--kg-construction-model",
    "openai:gpt-4.1-mini",
    "--kg-construction-workers",
    "3",
    "--candidate-beam-width",
    "96",
    "--max-candidates-per-question",
    "256",
    "--kg-cache-dir",
    "data/2WikiMultiHopQA_shared_v3/kg_cache",
    "--seed",
    "42",
]

if REBUILD_SPLITS:
    run(family_build if DATASET == "familyowl" else wiki_build)
else:
    print("REBUILD_SPLITS=False: using existing materialized split files.")

REBUILD_SPLITS=False: using existing materialized split files.


## Inspect split health

One row is one candidate subgraph, so summaries first collapse rows by question. For 2Wiki, `gold_kg_coverage` measures the upstream extraction ceiling. For FamilyOWL, `exact_candidate_available` shows whether candidate construction supplied a minimal proof.

In [9]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8-sig") as handle:
        return [json.loads(line) for line in handle if line.strip()]


split_rows = {
    split: read_jsonl(DATA_DIR / f"{split}_subgraph_retrieval.jsonl")
    for split in ("train", "dev", "test")
}
summary = []
for split, rows in split_rows.items():
    groups = {}
    for row in rows:
        groups.setdefault(row["example_id"], []).append(row)
    coverage = [
        float(
            group[0].get("gold_kg_coverage", group[0].get("gold_context_coverage", 0.0))
        )
        for group in groups.values()
    ]
    exact_available = [
        any(bool(row.get("exact_match_any_gold")) for row in group)
        for group in groups.values()
    ]
    rankable = [
        len(
            {
                round(
                    float(row.get("rank_target", row.get("best_set_f1_to_gold", 0.0))),
                    6,
                )
                for row in group
            }
        )
        > 1
        for group in groups.values()
    ]
    summary.append(
        {
            "split": split,
            "questions": len(groups),
            "candidate_rows": len(rows),
            "mean_upstream_coverage": sum(coverage) / max(len(coverage), 1),
            "complete_coverage_rate": sum(value >= 1.0 for value in coverage)
            / max(len(coverage), 1),
            "exact_candidate_rate": sum(exact_available) / max(len(exact_available), 1),
            "rankable_rate": sum(rankable) / max(len(rankable), 1),
        }
    )
pd.DataFrame(summary).set_index("split").round(3)

,questions,candidate_rows,mean_upstream_coverage,complete_coverage_rate,exact_candidate_rate,rankable_rate
split,,,,,,
train,64,17310,1.00,1.000,0.953,1.0
dev,32,8805,0.99,0.969,0.969,1.0
test,32,8854,1.00,1.000,1.000,1.0


In [10]:
# Change these to inspect a particular question and its oracle candidate ordering.
SPLIT = "test"
EXAMPLE_INDEX = 0

example_ids = list(dict.fromkeys(row["example_id"] for row in split_rows[SPLIT]))
example_id = example_ids[EXAMPLE_INDEX]
candidates = [row for row in split_rows[SPLIT] if row["example_id"] == example_id]
first = candidates[0]
print("ID:", example_id)
print("Question:", first.get("question"))
print("Answer (diagnostic only):", first.get("answer"))
print(
    "Gold KG/context coverage:",
    first.get("gold_kg_coverage", first.get("gold_context_coverage")),
)
print("Gold explanations:", first.get("gold_explanations"))
print("Matched gold KG triples:", first.get("matched_gold_kg_units", []))
print("Missing gold KG triples:", first.get("missing_gold_kg_units", []))
print(
    "Selected/all context sentences:",
    first.get("selected_context_sentence_count"),
    "/",
    first.get("context_sentence_count"),
)

candidate_table = pd.DataFrame(
    [
        {
            "oracle_f1": row.get("best_set_f1_to_gold", 0.0),
            "precision": row.get("best_set_precision_to_gold", 0.0),
            "recall": row.get("best_set_recall_to_gold", 0.0),
            "exact": row.get("exact_match_any_gold", False),
            "size": len(row.get("subgraph_units", [])),
            "candidate": " | ".join(row.get("subgraph_units", [])),
        }
        for row in candidates
    ]
).sort_values(["exact", "oracle_f1", "size"], ascending=[False, False, True])
candidate_table.head(20)

ID: FamilyOWL_1hop__g65__q0__1hop-Thing_james_alexander_bright_1921_james_alexander_bright_1921-james_alexander_bright_1921-rdf:type-Man-BIN__Is James Alexander Bright a man?
Question: Is James Alexander Bright a man?
Answer (diagnostic only): TRUE
Gold KG/context coverage: 1.0
Gold explanations: [['james_alexander_bright_1921 isBrotherOf patricia_bright_1938', 'isBrotherOf domain Man']]
Matched gold KG triples: []
Missing gold KG triples: []
Selected/all context sentences: None / None


,oracle_f1,precision,recall,exact,size,candidate
1,1.000000,1.000000,1.0,True,2,james_alexander_bright_1921 isBrotherOf patric...
3,0.800000,0.666667,1.0,False,3,james_alexander_bright_1921 isBrotherOf patric...
39,0.800000,0.666667,1.0,False,3,james_alexander_bright_1921 isBrotherOf patric...
40,0.800000,0.666667,1.0,False,3,james_alexander_bright_1921 isBrotherOf patric...
53,0.800000,0.666667,1.0,False,3,james_alexander_bright_1921 isBrotherOf patric...
115,0.800000,0.666667,1.0,False,3,james_alexander_bright_1921 isBrotherOf patric...
185,0.666667,1.000000,0.5,False,1,james_alexander_bright_1921 isBrotherOf patric...
34,0.500000,0.500000,0.5,False,2,hasMalePartner range Man | isBrotherOf domain Man
62,0.500000,0.500000,0.5,False,2,hasHusband range Man | isBrotherOf domain Man
70,0.500000,0.500000,0.5,False,2,james_alexander_bright_1921 isBrotherOf patric...


## Train and evaluate

Both datasets use the same coefficient-free graded listwise objective. Each candidate receives target probability proportional to its gold set F1, so exact proofs have the strongest individual target while useful partial evidence retains training signal. Checkpoints are selected by dev set F1@3, with precision@3 and exact MRR as tie-breakers.

In [ ]:
train_command = [
    PYTHON,
    "training/train_gnn_subgraph_retriever.py",
    "--train-path",
    DATA_DIR / "train_subgraph_retrieval.jsonl",
    "--dev-path",
    DATA_DIR / "dev_subgraph_retrieval.jsonl",
    "--save-dir",
    CHECKPOINT_DIR,
    "--epochs",
    str(EPOCHS),
    "--candidate-batch-size",
    "256",
    "--architecture-version",
    "3",
    "--freeze-encoder",
]
if TRAIN_MODEL:
    run(train_command)
else:
    print("TRAIN_MODEL=False: command is ready but was not executed.")

eval_command = [
    PYTHON,
    "evaluation/eval_gnn_subgraph_retriever.py",
    "--train-path",
    DATA_DIR / "train_subgraph_retrieval.jsonl",
    "--dev-path",
    DATA_DIR / "dev_subgraph_retrieval.jsonl",
    "--test-path",
    DATA_DIR / "test_subgraph_retrieval.jsonl",
    "--checkpoint",
    CHECKPOINT_DIR / "best_model.pt",
    "--candidate-batch-size",
    "256",
    "--save-details",
    "--details-dir",
    DETAILS_DIR,
]
if EVALUATE_MODEL:
    run(eval_command)
else:
    print("EVALUATE_MODEL=False: command is ready but was not executed.")

> c:\Users\julie\github\PhD\SAGE-QA\.venv\Scripts\python.exe training/train_gnn_subgraph_retriever.py --train-path c:\Users\julie\github\PhD\SAGE-QA\data\development\gnn_rebuilt\FamilyOWL_1hop\train_subgraph_retrieval.jsonl --dev-path c:\Users\julie\github\PhD\SAGE-QA\data\development\gnn_rebuilt\FamilyOWL_1hop\dev_subgraph_retrieval.jsonl --save-dir c:\Users\julie\github\PhD\SAGE-QA\checkpoints\development\workbench\familyowl_graded_listwise --epochs 5 --candidate-batch-size 256 --architecture-version 3 --freeze-encoder
> c:\Users\julie\github\PhD\SAGE-QA\.venv\Scripts\python.exe evaluation/eval_gnn_subgraph_retriever.py --train-path c:\Users\julie\github\PhD\SAGE-QA\data\development\gnn_rebuilt\FamilyOWL_1hop\train_subgraph_retrieval.jsonl --dev-path c:\Users\julie\github\PhD\SAGE-QA\data\development\gnn_rebuilt\FamilyOWL_1hop\dev_subgraph_retrieval.jsonl --test-path c:\Users\julie\github\PhD\SAGE-QA\data\development\gnn_rebuilt\FamilyOWL_1hop\test_subgraph_retrieval.jsonl --checkpoi

## Read metrics and detailed model failures

The evaluator calculates these metrics over all questions. `exact_hit@k` is the fraction with an exact proof in the top *k*. `exact_mrr` averages `1 / best_exact_rank` over questions where an exact candidate exists. Precision, recall, and set F1 compare retrieved evidence units with the gold proof. The failure table then separates upstream candidate failures from ranking failures.

In [ ]:
metric_path = DETAILS_DIR / "test_metrics.json"
if metric_path.exists():
    metrics = json.loads(metric_path.read_text(encoding="utf-8"))
    important = [
        "exact_candidate_rate",
        "exact_mrr",
        "exact_hit@1",
        "exact_hit@3",
        "best_precision@1",
        "best_recall@1",
        "best_set_f1@1",
        "precision@3",
        "recall@3",
        "set_f1@3",
        "collapsed_score_rate",
    ]
    display(
        pd.DataFrame(
            {"metric": important, "value": [metrics.get(name) for name in important]}
        )
    )
else:
    print("No test metrics found yet in", metric_path)

detail_files = (
    sorted(DETAILS_DIR.glob("*details*.json")) if DETAILS_DIR.exists() else []
)
if detail_files:
    details = json.loads(detail_files[-1].read_text(encoding="utf-8"))
    columns = [
        "example_id",
        "failure_mode",
        "gold_kg_coverage",
        "exact_candidate_available",
        "best_exact_rank",
        "top1_best_set_f1_to_gold",
        "candidate_score_std",
    ]
    display(
        pd.DataFrame(details)[
            [column for column in columns if column in pd.DataFrame(details).columns]
        ].sort_values(["failure_mode", "example_id"])
    )
else:
    print("No evaluation details found yet in", DETAILS_DIR)

,metric,value
0,exact_candidate_rate,1.000000
1,exact_mrr,0.245107
2,exact_hit@1,0.093750
3,exact_hit@3,0.312500
4,best_precision@1,0.265625
5,best_recall@1,0.291667
6,best_set_f1@1,0.268750
7,precision@3,0.275000
8,recall@3,0.630208
9,set_f1@3,0.371205


,example_id,failure_mode,gold_kg_coverage,exact_candidate_available,best_exact_rank,top1_best_set_f1_to_gold,candidate_score_std
5,FamilyOWL_1hop__g31__q0__1hop-Thing_eliza_brig...,candidate_generation_no_exact,0.0,False,NaN,0.4,0.105501
7,FamilyOWL_1hop__g55__q0__1hop-Thing_harriet_br...,candidate_generation_no_exact,0.0,False,NaN,0.4,0.133199
8,FamilyOWL_1hop__g77__q0__1hop-Thing_james_tubb...,candidate_generation_no_exact,0.0,False,NaN,0.8,0.077829
13,FamilyOWL_1hop__g333__q3__1hop-Thing_alec_john...,pass_exact,0.0,True,1.0,1.0,0.088780
17,FamilyOWL_1hop__g366__q1__1hop-Thing_charles_j...,pass_exact,0.0,True,1.0,1.0,0.030967
...,...,...,...,...,...,...,...
59,FamilyOWL_1hop__g709__q1__1hop-Thing_susannah_...,ranking_exact_available,0.0,True,4.0,0.0,0.040215
61,FamilyOWL_1hop__g734__q3__1hop-Thing_william_a...,ranking_exact_available,0.0,True,2.0,0.0,0.054762
63,FamilyOWL_1hop__g743__q0__1hop-Thing_william_i...,ranking_exact_available,0.0,True,23.0,0.5,0.019739
2,FamilyOWL_1hop__g7__q0__1hop-Thing_annie_whitf...,ranking_exact_available,0.0,True,7.0,0.4,0.048140
